[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Testing and Packaging](https://johnfisher-ai.github.io/Python-Visual-Guides/testing-and-packaging.html)

# Test Structure


## What you will be able to do

Write a test in three steps, arrange, act and assert, name it for the behavior it checks, and keep it
to one behavior, so that a failure says what broke and hides nothing else. Keep a project's tests in
a `tests` folder, grouped in files and classes, and write tests that pass alone and in any order.


## The idea

### The problem

The tests in the **Your First Test** notebook worked, and a test file that keeps growing tends to
lose its shape. One test checks five things, and when its second check fails, nobody learns whether
the other three still hold. Another is called `test_mean2`, and its failure says only that something
about `mean` broke. A list at the top of the file is changed by one test and read by another, so the
second test passes when it runs alone and fails when the whole file runs. The lines for a day of
readings are copied into every test, a little differently each time.

None of that stops a test from passing on the day it is written. All of it makes a failure harder to
read on the day a failure comes, and that day is the reason the test exists.

### What a well-structured test is

> A test takes three steps. It **arranges** the situation the code will meet, **acts** by running
> the code under test, usually with a single call, and **asserts** what that call produced. pytest's
> documentation names a fourth step, **cleanup**, which undoes anything the test changed outside
> itself. A well-structured test checks **one behavior**, has a name that states that behavior, and
> is **independent**: it passes when it runs alone, in any order, and beside any other test.

### Why it works that way

- **A failure is read from its name first.**
  `FAILED tests/test_mean.py::TestMean::test_of_no_readings_is_none` says what broke before anybody
  opens a file, and `FAILED test_readings.py::test_mean2` says only where to start looking.
- **A test stops at its first failing `assert`.** A test of five behaviors reports the first behavior
  that broke and hides the other four, while five tests report every behavior that broke, and every
  one that still works.
- **The same three steps in every test make any test quick to read.** The input, the call and the
  check are in the same places every time.
- **Independence is what lets you run part of a suite.** `-k`, `--lf` and a node ID all run some
  tests without the others, and a test that needed another test's side effect fails exactly then.
- **Files and folders are structure too.** A test file for each part of the code, in a `tests`
  folder, lets you run the tests of one part, and keeps test code apart from the code a project
  ships.

### Where this shows up

pytest's documentation describes the anatomy of a test in these steps, and the pattern has other
names elsewhere, such as given, when and then in behavior-driven development. pytest's guide to good
integration practices suggests keeping tests in a `tests` folder beside the code, which the
**Project Layout** notebook sets up in a project that installs. The **Fixtures** notebook moves the
arrange step out of a test and into a function pytest runs for any test that asks for it, and the
**Parametrize** notebook runs one test's act and assert over many arranged inputs.

### What this notebook covers

- Arrange, act and assert, marked in a test
- Names that state a behavior, and a report that reads as a list of what the code does
- One behavior per test, and the failure that a test of several behaviors hides
- Several asserts about one behavior, and a whole value compared in one `assert`
- A `tests` folder, with a test file for each part of the module
- A class that groups the tests of one function
- A helper function that arranges test data
- The restructured suite, and a bug it names exactly
- Five errors, from a test that changes another test's data to two test files with one name

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import subprocess
import sys
from pathlib import Path

Path("test_structure.py").write_text('''
def to_fahrenheit(celsius):
    return celsius * 2 + 30                      # a rough rule, right only at 10 degrees


def test_conversions():
    assert to_fahrenheit(10) == 50
    assert to_fahrenheit(0) == 32
    assert to_fahrenheit(100) == 212


def test_ten_degrees():
    assert to_fahrenheit(10) == 50


def test_freezing_point():
    assert to_fahrenheit(0) == 32


def test_boiling_point():
    assert to_fahrenheit(100) == 212
''')

finished = subprocess.run([sys.executable, "-m", "pytest", "-q", "--tb=no"],
                          capture_output=True, text=True)
print(finished.stdout, end="")
```

```
F.FF                                                                     [100%]
=========================== short test summary info ============================
FAILED test_structure.py::test_conversions - assert 30 == 32
FAILED test_structure.py::test_freezing_point - assert 30 == 32
FAILED test_structure.py::test_boiling_point - assert 230 == 212
3 failed, 1 passed in 0.01s
```

The same three checks, written as one test and as three tests, against a conversion that is right
only at 10 degrees. The one test failed at its second `assert` and never reached its third, so its
line in the summary mentions the freezing point and hides that the boiling point is wrong too. The
three tests report both failures and the conversion that still works, and their names say which is
which. `--tb=no` leaves out the details of each failure, which leaves the summary.


## Setup

Six imports, and the functions that run pytest.

- `subprocess` runs pytest as a program of its own, in `run_pytest`
- `sys` names the Python that runs it
- `os` passes pytest the environment, with `NO_COLOR` and `PYTHONDONTWRITEBYTECODE` set in it, as in
  the **Your First Test** notebook
- `re` takes out of pytest's report the parts that differ between computers
- `Path` makes the project's folders, and moves and lists the files in them
- `shutil` removes the scratch folder at the end

`pytest_report` and `run_pytest` are the functions the **Your First Test** notebook wrote, which run
`python -m pytest` in the project's folder, `scratch/stations`, and that notebook explains each of
their settings.


In [1]:
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

PROJECT = Path("scratch/stations")
PROJECT.mkdir(parents=True, exist_ok=True)
os.environ["NO_COLOR"] = "1"                  # programs started from here print without color codes
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"   # and keep no compiled copies, which a quick rewrite can outrun


def pytest_report(*arguments, folder=PROJECT):
    """What python -m pytest prints when it runs in the folder, less what differs between computers."""
    settings = {"COLUMNS": "80", "PYTEST_DISABLE_PLUGIN_AUTOLOAD": "1", "PYTHONNODEBUGRANGES": "1"}
    finished = subprocess.run([sys.executable, "-m", "pytest", "--no-header", *arguments],
                              cwd=folder, capture_output=True, text=True, env={**os.environ, **settings})
    report = finished.stdout + finished.stderr
    report = report.replace(f"{Path(folder).resolve()}/", "")          # the folder's own path
    report = re.sub(r"\S*/_pytest/", "_pytest/", report)                # the path to pytest's own files
    return re.sub(r" in \d+\.\d+s\b", "", report).rstrip()              # the time the run took


def run_pytest(*arguments, folder=PROJECT):
    """Run python -m pytest in the folder, as a terminal would, and print its report."""
    print(pytest_report(*arguments, folder=folder))


print("ready:", PROJECT)


ready: scratch/stations


## Worked examples

### Arrange, act, assert

The project's module is the one the **Your First Test** notebook finished with:


In [2]:
%%writefile scratch/stations/readings.py
"""Readings from the weather stations, and each station's mean temperature."""

import statistics


def parse_reading(line):
    """A (station, celsius) pair from a line such as 'Bergen,4.2'. An empty reading is None.

    A minus sign written as U+2212, as some spreadsheets write it, reads as a hyphen-minus.
    """
    station, celsius = line.strip().split(",")
    celsius = celsius.replace("\u2212", "-")
    return station, float(celsius) if celsius else None


def mean(values):
    """The mean of the readings that are not None, or None when there are none."""
    present = [value for value in values if value is not None]
    return statistics.fmean(present) if present else None


def summarize(lines):
    """Each station's mean temperature, from lines of readings. A blank line is skipped."""
    by_station = {}
    for line in lines:
        if not line.strip():
            continue
        station, celsius = parse_reading(line)
        by_station.setdefault(station, []).append(celsius)
    return {station: mean(values) for station, values in by_station.items()}


def to_fahrenheit(celsius):
    """A temperature in degrees Celsius, in degrees Fahrenheit."""
    return celsius * 9 / 5 + 32


Writing scratch/stations/readings.py


Here is a test of `summarize`, with its three steps marked by comments:


In [3]:
%%writefile scratch/stations/test_summary.py
from readings import summarize


def test_a_station_with_only_empty_readings_has_no_mean():
    # arrange: a day on which Svalbard sent only empty readings
    lines = ["Bergen,4.2", "Svalbard,", "Svalbard,"]

    # act: run the code under test, once
    summary = summarize(lines)

    # assert: check what it produced
    assert summary["Svalbard"] is None


Writing scratch/stations/test_summary.py


In [4]:
run_pytest("-v")


============================= test session starts ==============================
collecting ... collected 1 item

test_summary.py::test_a_station_with_only_empty_readings_has_no_mean PASSED [100%]

============================== 1 passed ===============================


The arrange step builds the day, the act step makes one call and keeps its result, and the assert
step checks that result. pytest's documentation names a fourth step, cleanup, for a test that
changes something outside itself, such as a file on disk, which the **Fixtures** notebook handles.
The comments name the steps in this first test. Most tests mark the steps with blank lines alone, as
the rest of this notebook does.

### A name that states the behavior

A test's name is the first thing its failure shows, so a name that states a behavior says what broke
before anybody opens the file:

| A name that says where to look | A name that says what broke |
|---|---|
| `test_mean` | `test_mean_skips_missing_readings` |
| `test_mean2` | `test_mean_of_no_readings_is_none` |
| `test_parse` | `test_an_empty_reading_is_none` |
| `test_bug_fix` | `test_a_blank_line_is_skipped` |

With names like those, `-v` prints a list of what the module does:


In [5]:
%%writefile scratch/stations/test_readings.py
from readings import mean, parse_reading, to_fahrenheit


def test_a_reading_is_a_station_and_a_temperature():
    assert parse_reading("Bergen,4.2") == ("Bergen", 4.2)


def test_an_empty_reading_is_none():
    assert parse_reading("Svalbard,") == ("Svalbard", None)


def test_mean_of_two_readings():
    assert mean([4.2, 5.8]) == 5.0


def test_mean_skips_missing_readings():
    assert mean([4.2, None, 5.8]) == 5.0


def test_mean_of_no_readings_is_none():
    assert mean([None, None]) is None


def test_freezing_point_in_fahrenheit():
    assert to_fahrenheit(0) == 32


def test_boiling_point_in_fahrenheit():
    assert to_fahrenheit(100) == 212


Writing scratch/stations/test_readings.py


In [6]:
run_pytest("test_readings.py", "-v")


============================= test session starts ==============================
collecting ... collected 7 items

test_readings.py::test_a_reading_is_a_station_and_a_temperature PASSED   [ 14%]
test_readings.py::test_an_empty_reading_is_none PASSED                   [ 28%]
test_readings.py::test_mean_of_two_readings PASSED                       [ 42%]
test_readings.py::test_mean_skips_missing_readings PASSED                [ 57%]
test_readings.py::test_mean_of_no_readings_is_none PASSED                [ 71%]
test_readings.py::test_freezing_point_in_fahrenheit PASSED               [ 85%]
test_readings.py::test_boiling_point_in_fahrenheit PASSED                [100%]

============================== 7 passed ===============================


Read the names down the report and you have the module's behaviors, every one of them checked. A
long name costs little: nobody types it in full, since `-k` matches part of a name, and the report
has room for it.

### One behavior per test

A test stops at its first failing `assert`, so a test that checks several behaviors reports the first
one that broke and says nothing about the rest. Here is `parse_reading`, checked four ways in one
test:


In [7]:
%%writefile scratch/stations/test_parsing.py
from readings import parse_reading


def test_parse_reading():
    assert parse_reading("Bergen,4.2") == ("Bergen", 4.2)
    assert parse_reading("Svalbard,") == ("Svalbard", None)
    assert parse_reading("Tromso,\u22126.3") == ("Tromso", -6.3)
    assert parse_reading(" Oslo,-2.4 ") == ("Oslo", -2.4)


Writing scratch/stations/test_parsing.py


In [8]:
run_pytest("test_parsing.py", "-q")


.                                                                        [100%]
1 passed


It passes. Now somebody shortens `parse_reading`, and two of its behaviors go with the lines that
disappear: spaces around a line are no longer stripped, and U+2212 is no longer replaced. This cell
keeps the module's text first, to put it back later:


In [9]:
module = (PROJECT / "readings.py").read_text()          # the module as it is, to put back later

print(len(module.splitlines()), "lines kept")


35 lines kept


In [10]:
%%writefile scratch/stations/readings.py
"""Readings from the weather stations, and each station's mean temperature."""

import statistics


def parse_reading(line):
    """A (station, celsius) pair from a line such as 'Bergen,4.2'. An empty reading is None."""
    station, celsius = line.split(",")
    return station, float(celsius) if celsius else None


def mean(values):
    """The mean of the readings that are not None, or None when there are none."""
    present = [value for value in values if value is not None]
    return statistics.fmean(present) if present else None


def summarize(lines):
    """Each station's mean temperature, from lines of readings. A blank line is skipped."""
    by_station = {}
    for line in lines:
        if not line.strip():
            continue
        station, celsius = parse_reading(line)
        by_station.setdefault(station, []).append(celsius)
    return {station: mean(values) for station, values in by_station.items()}


def to_fahrenheit(celsius):
    """A temperature in degrees Celsius, in degrees Fahrenheit."""
    return celsius * 9 / 5 + 32


Overwriting scratch/stations/readings.py


In [11]:
run_pytest("test_parsing.py", "-q")


F                                                                        [100%]
=================================== FAILURES ===================================
______________________________ test_parse_reading ______________________________

    def test_parse_reading():
        assert parse_reading("Bergen,4.2") == ("Bergen", 4.2)
        assert parse_reading("Svalbard,") == ("Svalbard", None)
>       assert parse_reading("Tromso,\u22126.3") == ("Tromso", -6.3)

test_parsing.py:7: 
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ 

line = 'Tromso,−6.3'

    def parse_reading(line):
        """A (station, celsius) pair from a line such as 'Bergen,4.2'. An empty reading is None."""
        station, celsius = line.split(",")
>       return station, float(celsius) if celsius else None
E       ValueError: could not convert string to float: '−6.3'

readings.py:9: ValueError
=========================== short test summary info ============================
FAILE

The report is about U+2212, and only about it. The test stopped at its third `assert`, and whether a
line with spaces around it still parses stays unknown until this failure is fixed and the suite runs
again. The same four checks, as four tests:


In [12]:
%%writefile scratch/stations/test_parsing.py
from readings import parse_reading


def test_a_reading_is_a_station_and_a_temperature():
    assert parse_reading("Bergen,4.2") == ("Bergen", 4.2)


def test_an_empty_reading_is_none():
    assert parse_reading("Svalbard,") == ("Svalbard", None)


def test_a_unicode_minus_sign_reads_as_negative():
    assert parse_reading("Tromso,\u22126.3") == ("Tromso", -6.3)


def test_spaces_around_a_line_are_ignored():
    assert parse_reading(" Oslo,-2.4 ") == ("Oslo", -2.4)


Overwriting scratch/stations/test_parsing.py


In [13]:
run_pytest("test_parsing.py", "-q")


..FF                                                                     [100%]
=================================== FAILURES ===================================
_________________ test_a_unicode_minus_sign_reads_as_negative __________________

    def test_a_unicode_minus_sign_reads_as_negative():
>       assert parse_reading("Tromso,\u22126.3") == ("Tromso", -6.3)

test_parsing.py:13: 
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ 

line = 'Tromso,−6.3'

    def parse_reading(line):
        """A (station, celsius) pair from a line such as 'Bergen,4.2'. An empty reading is None."""
        station, celsius = line.split(",")
>       return station, float(celsius) if celsius else None
E       ValueError: could not convert string to float: '−6.3'

readings.py:9: ValueError
____________________ test_spaces_around_a_line_are_ignored _____________________

    def test_spaces_around_a_line_are_ignored():
>       assert parse_reading(" Oslo,-2.4 ") == ("Oslo",

Two failures and two passes, each under a name that says which behavior it is. One change broke two
behaviors, and one run reports both, along with the two that still work. The module goes back as it
was:


In [14]:
(PROJECT / "readings.py").write_text(module)

run_pytest("-q")


............                                                             [100%]
12 passed


### Several asserts about one behavior, or one assert about a whole value

One behavior is not the same as one `assert`. Tuesday's summary is one behavior, and a test can check
it station by station, with four asserts, or as a whole value, with one. The two tests differ when
something breaks. Here are both, with Tuesday's lines in a function, `tuesday`:


In [15]:
%%writefile scratch/stations/test_summary.py
from readings import summarize


def tuesday():
    """Tuesday's lines of readings, as the stations sent them."""
    return ["Bergen,4.2", "Bergen,5.8", "Bergen,", "Oslo,-2.4", "Oslo,-1.6",
            "Svalbard,", "Svalbard,", "Tromso,-6.3", "Tromso,-5.7", "Tromso,"]


def test_a_station_with_only_empty_readings_has_no_mean():
    lines = ["Bergen,4.2", "Svalbard,", "Svalbard,"]

    summary = summarize(lines)

    assert summary["Svalbard"] is None


def test_summary_of_tuesday_station_by_station():
    summary = summarize(tuesday())

    assert summary["Bergen"] == 5.0
    assert summary["Oslo"] == -2.0
    assert summary["Svalbard"] is None
    assert summary["Tromso"] == -6.0


def test_summary_of_tuesday():
    summary = summarize(tuesday())

    assert summary == {"Bergen": 5.0, "Oslo": -2.0, "Svalbard": None, "Tromso": -6.0}


Overwriting scratch/stations/test_summary.py


Now `mean` changes, so that a station with no readings gets `0.0` instead of `None`:


In [16]:
(PROJECT / "readings.py").write_text(module.replace("if present else None", "if present else 0.0"))

run_pytest("test_summary.py", "-q")


FFF                                                                      [100%]
=================================== FAILURES ===================================
_____________ test_a_station_with_only_empty_readings_has_no_mean ______________

    def test_a_station_with_only_empty_readings_has_no_mean():
        lines = ["Bergen,4.2", "Svalbard,", "Svalbard,"]
    
        summary = summarize(lines)
    
>       assert summary["Svalbard"] is None
E       assert 0.0 is None

test_summary.py:15: AssertionError
__________________ test_summary_of_tuesday_station_by_station __________________

    def test_summary_of_tuesday_station_by_station():
        summary = summarize(tuesday())
    
        assert summary["Bergen"] == 5.0
        assert summary["Oslo"] == -2.0
>       assert summary["Svalbard"] is None
E       assert 0.0 is None

test_summary.py:23: AssertionError
___________________________ test_summary_of_tuesday ____________________________

    def test_summary_of_tuesday():
    

The station-by-station test stopped at Svalbard, the third of its four asserts, so its report cannot
say whether Tromso is right. The whole-value test compared all four stations in one `assert`, and its
report says that three items were identical and which one differed. When a test can state all of a
value, comparing the whole value reports every difference in one run. The module goes back:


In [17]:
(PROJECT / "readings.py").write_text(module)

run_pytest("-q")


..............                                                           [100%]
14 passed


### A tests folder, with a file for each part of the module

Test files beside the module suit a small project. A project that grows keeps its tests in a folder
of their own, named `tests`, with a test file for each part of the code. The parsing tests already
have `test_parsing.py`, and the summary tests `test_summary.py`, so those two files move. The rest
of `test_readings.py` splits into a file for `mean` and a file for the conversions, and the file
itself goes, since its parsing tests are in `test_parsing.py` now:


In [18]:
tests = PROJECT / "tests"
tests.mkdir(exist_ok=True)
for name in ["test_parsing.py", "test_summary.py"]:
    (PROJECT / name).rename(tests / name)
(PROJECT / "test_readings.py").unlink()

print(sorted(path.name for path in tests.iterdir()))


['test_parsing.py', 'test_summary.py']


In [19]:
%%writefile scratch/stations/tests/test_mean.py
from readings import mean


def test_mean_of_two_readings():
    assert mean([4.2, 5.8]) == 5.0


def test_mean_skips_missing_readings():
    assert mean([4.2, None, 5.8]) == 5.0


def test_mean_of_no_readings_is_none():
    assert mean([None, None]) is None


Writing scratch/stations/tests/test_mean.py


In [20]:
%%writefile scratch/stations/tests/test_conversions.py
import pytest

from readings import to_fahrenheit


def test_freezing_point_in_fahrenheit():
    assert to_fahrenheit(0) == 32


def test_boiling_point_in_fahrenheit():
    assert to_fahrenheit(100) == 212


def test_body_temperature_in_fahrenheit():
    assert to_fahrenheit(37) == pytest.approx(98.6)


Writing scratch/stations/tests/test_conversions.py


In [21]:
print(sorted(str(path.relative_to(PROJECT)) for path in PROJECT.rglob("*.py")))

run_pytest("-q")


['readings.py', 'tests/test_conversions.py', 'tests/test_mean.py', 'tests/test_parsing.py', 'tests/test_summary.py']
.............                                                            [100%]
13 passed


`python -m pytest`, run in the project's folder, found the tests in `tests`, and they still import
`readings`, because `python -m` puts the folder it runs in on the import path, as the **Your First
Test** notebook said. The **Project Layout** notebook shows why a project should not rely on that,
and what it does instead. A node ID now starts with the folder, and a path runs one part of the
suite:


In [22]:
run_pytest("tests/test_mean.py", "-v")


============================= test session starts ==============================
collecting ... collected 3 items

tests/test_mean.py::test_mean_of_two_readings PASSED                     [ 33%]
tests/test_mean.py::test_mean_skips_missing_readings PASSED              [ 66%]
tests/test_mean.py::test_mean_of_no_readings_is_none PASSED              [100%]

============================== 3 passed ===============================


### A class for the tests of one function

Inside a file, a class can group the tests of one function. The class's name starts with `Test`, it
has no `__init__`, and every test is a method that takes `self`. A test's name can then leave out the
function's name, since the class's name carries it:


In [23]:
%%writefile scratch/stations/tests/test_mean.py
from readings import mean


class TestMean:
    def test_of_two_readings(self):
        assert mean([4.2, 5.8]) == 5.0

    def test_skips_missing_readings(self):
        assert mean([4.2, None, 5.8]) == 5.0

    def test_of_no_readings_is_none(self):
        assert mean([None, None]) is None

    def test_of_one_reading_is_that_reading(self):
        assert mean([-6.3]) == -6.3


Overwriting scratch/stations/tests/test_mean.py


In [24]:
run_pytest("tests/test_mean.py", "-v")
print()
run_pytest("-k", "TestMean and none", "-v")


============================= test session starts ==============================
collecting ... collected 4 items

tests/test_mean.py::TestMean::test_of_two_readings PASSED                [ 25%]
tests/test_mean.py::TestMean::test_skips_missing_readings PASSED         [ 50%]
tests/test_mean.py::TestMean::test_of_no_readings_is_none PASSED         [ 75%]
tests/test_mean.py::TestMean::test_of_one_reading_is_that_reading PASSED [100%]

============================== 4 passed ===============================

============================= test session starts ==============================
collecting ... collected 14 items / 13 deselected / 1 selected

tests/test_mean.py::TestMean::test_of_no_readings_is_none PASSED         [100%]

======================= 1 passed, 13 deselected =======================


The node ID joins the file, the class and the method, and `-k` matches a class's name as well as a
test's. pytest makes a new instance of the class for every test, so a value one test stores on `self`
is gone when the next test runs. The class groups tests, and shares nothing between them.

### A helper that arranges test data

The arrange step of a summary test writes out lines of readings, which crowd the one thing that
matters in each test. A helper function in the test file can build the lines, and since its name
does not start with `test`, pytest leaves it alone:


In [25]:
%%writefile scratch/stations/tests/test_summary.py
from readings import summarize


def lines_for(station, *readings):
    """Lines of readings from one station, with None for an empty reading."""
    return [f"{station},{'' if reading is None else reading}" for reading in readings]


def tuesday():
    """Tuesday's lines of readings, as the stations sent them."""
    return (lines_for("Bergen", 4.2, 5.8, None) + lines_for("Oslo", -2.4, -1.6)
            + lines_for("Svalbard", None, None) + lines_for("Tromso", -6.3, -5.7, None))


def test_a_station_with_only_empty_readings_has_no_mean():
    lines = lines_for("Bergen", 4.2) + lines_for("Svalbard", None, None)

    summary = summarize(lines)

    assert summary["Svalbard"] is None


def test_a_blank_line_is_skipped():
    lines = lines_for("Bergen", 4.2) + [""] + lines_for("Bergen", 5.8)

    summary = summarize(lines)

    assert summary == {"Bergen": 5.0}


def test_summary_of_tuesday():
    summary = summarize(tuesday())

    assert summary == {"Bergen": 5.0, "Oslo": -2.0, "Svalbard": None, "Tromso": -6.0}


Overwriting scratch/stations/tests/test_summary.py


In [26]:
run_pytest("tests/test_summary.py", "-v")


============================= test session starts ==============================
collecting ... collected 3 items

tests/test_summary.py::test_a_station_with_only_empty_readings_has_no_mean PASSED [ 33%]
tests/test_summary.py::test_a_blank_line_is_skipped PASSED               [ 66%]
tests/test_summary.py::test_summary_of_tuesday PASSED                    [100%]

============================== 3 passed ===============================


`lines_for("Svalbard", None, None)` says in a few words what two lines of text said before, and every
call to `lines_for` or `tuesday` returns a new list, so no test can change the lines another test
receives. A list written once at the top of a file cannot promise that, as Common errors shows. The
**Fixtures** notebook turns a helper like `tuesday` into a fixture, which a test asks for by name.

### The restructured suite, and a bug it names

The pieces of this notebook, in one suite: tests in a `tests` folder, a file for each part of the
module, a class for the tests of `mean`, a helper for the summary tests, and names that state
behaviors. With `-v`, the report lists what the module does:


In [27]:
run_pytest("-v")


============================= test session starts ==============================
collecting ... collected 14 items

tests/test_conversions.py::test_freezing_point_in_fahrenheit PASSED      [  7%]
tests/test_conversions.py::test_boiling_point_in_fahrenheit PASSED       [ 14%]
tests/test_conversions.py::test_body_temperature_in_fahrenheit PASSED    [ 21%]
tests/test_mean.py::TestMean::test_of_two_readings PASSED                [ 28%]
tests/test_mean.py::TestMean::test_skips_missing_readings PASSED         [ 35%]
tests/test_mean.py::TestMean::test_of_no_readings_is_none PASSED         [ 42%]
tests/test_mean.py::TestMean::test_of_one_reading_is_that_reading PASSED [ 50%]
tests/test_parsing.py::test_a_reading_is_a_station_and_a_temperature PASSED [ 57%]
tests/test_parsing.py::test_an_empty_reading_is_none PASSED              [ 64%]
tests/test_parsing.py::test_a_unicode_minus_sign_reads_as_negative PASSED [ 71%]
tests/test_parsing.py::test_spaces_around_a_line_are_ignored PASSED      [ 78%]


Now a bug: a rewrite of `summarize` loses the lines that skip a blank line. `--tb=no` leaves out the
details, to show what the summary alone says:


In [28]:
(PROJECT / "readings.py").write_text(module.replace("        if not line.strip():\n            continue\n", ""))

run_pytest("-q", "--tb=no")


............F.                                                           [100%]
=========================== short test summary info ============================
FAILED tests/test_summary.py::test_a_blank_line_is_skipped - ValueError: not ...
1 failed, 13 passed


One line of the report names the folder, the file, the behavior that broke and the exception, and
the thirteen tests that passed say that every other behavior of the module still holds. The module
goes back:


In [29]:
(PROJECT / "readings.py").write_text(module)

run_pytest("-q")


..............                                                           [100%]
14 passed


### Where each part came from

| In the suite | What it relies on | The section that showed it |
|---|---|---|
| blank lines between the steps of a test | arrange, act and assert, in that order | Arrange, act, assert |
| `test_a_blank_line_is_skipped`, in the failure's line | a name that states the behavior | A name that states the behavior |
| one test for each way `parse_reading` reads a line | a test stops at its first failing `assert` | One behavior per test |
| `assert summary == {...}` | a whole value reports every difference | Several asserts about one behavior, or one assert about a whole value |
| `tests/test_mean.py`, `tests/test_summary.py` | a file for each part, run on its own by path | A tests folder, with a file for each part of the module |
| `TestMean` | a class that groups tests and shares nothing | A class for the tests of one function |
| `lines_for` and `tuesday` | a new list of lines for every test | A helper that arranges test data |

A failure in this suite reads as a sentence about the module, and every test in it can run alone,
which is what `-k`, `--lf` and a node ID need.


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/testing-and-packaging/03-test-structure-solutions.ipynb).

**1.** Write `tests/test_tasks.py` with a test of `to_fahrenheit` in three steps, separated by blank
lines: arrange the temperature `-40`, act, and assert that the result is `-40`. Run the file with
`-v`.


In [30]:
# your code here


**2.** This test checks three behaviors:

```python
def test_mean():
    assert mean([4.2, 5.8]) == 5.0
    assert mean([None]) is None
    assert mean([-6.3]) == -6.3
```

Write it into `tests/test_tasks.py` as three tests whose names say what each one checks, keeping the
test from task 1, and run the file with `-v`.


In [31]:
# your code here


**3.** Group the three tests of task 2 in a class, `TestMeanTasks`, and run only that class with
`-k`.


In [32]:
# your code here


**4.** Run one test of `TestMeanTasks` by its node ID.


In [33]:
# your code here


**5.** These two tests pass when they run together, in this order, and one of them fails alone:

```python
STATIONS = []


def test_adding_a_station():
    STATIONS.append("Alta")
    assert STATIONS == ["Alta"]


def test_the_first_station_is_alta():
    assert STATIONS[0] == "Alta"
```

Write them into `tests/test_order.py`, run the file, and run the second test alone. Then change the
tests so that each one passes alone and together.


In [34]:
# your code here


**6.** Write `tests/test_two_stations.py` with a helper, `lines_for`, like the one in
`tests/test_summary.py`, and a test that uses it to check the summary of two stations with one
reading each, `4.2` from Bergen and `-2.4` from Oslo. Run the file.


In [35]:
# your code here


## Common errors

### assert 6.0 == 5.0


In [36]:
%%writefile scratch/stations/tests/test_shared.py
from readings import mean

READINGS = [4.2, 5.8]


def test_a_third_reading_raises_the_mean():
    READINGS.append(8.0)
    assert mean(READINGS) == 6.0


def test_mean_of_the_readings():
    assert mean(READINGS) == 5.0


Writing scratch/stations/tests/test_shared.py


In [37]:
run_pytest("tests/test_shared.py", "-q")
print()
run_pytest("tests/test_shared.py::test_mean_of_the_readings", "-q")


.F                                                                       [100%]
=================================== FAILURES ===================================
__________________________ test_mean_of_the_readings ___________________________

    def test_mean_of_the_readings():
>       assert mean(READINGS) == 5.0
E       assert 6.0 == 5.0
E        +  where 6.0 = mean([4.2, 5.8, 8.0])

tests/test_shared.py:12: AssertionError
=========================== short test summary info ============================
FAILED tests/test_shared.py::test_mean_of_the_readings - assert 6.0 == 5.0
1 failed, 1 passed

.                                                                        [100%]
1 passed


The second test fails when the file runs and passes when it runs alone. `READINGS` is one list for
the whole file, and the first test appended to it, so by the time the second test ran, the list held
three readings, as the `where` line shows. A test that changes shared data breaks other tests, and
only in some runs, which makes the failure hard to trace. Let every test arrange its own list:


In [38]:
%%writefile scratch/stations/tests/test_shared.py
from readings import mean


def test_a_third_reading_raises_the_mean():
    readings = [4.2, 5.8, 8.0]

    assert mean(readings) == 6.0


def test_mean_of_the_readings():
    readings = [4.2, 5.8]

    assert mean(readings) == 5.0


Overwriting scratch/stations/tests/test_shared.py


In [39]:
run_pytest("tests/test_shared.py", "-q")


..                                                                       [100%]
2 passed


### No error, and a test that vanished: two tests with one name


In [40]:
%%writefile scratch/stations/tests/test_twice.py
from readings import to_fahrenheit


def test_a_conversion():
    assert to_fahrenheit(0) == 32


def test_a_conversion():
    assert to_fahrenheit(100) == 212


Writing scratch/stations/tests/test_twice.py


In [41]:
run_pytest("tests/test_twice.py", "--collect-only", "-q")


tests/test_twice.py::test_a_conversion

1 test collected


Two tests in the file, and one collected. The second `def` made a new function under the same name,
and the first function was gone before pytest looked, so the check at 0 degrees never runs and
nothing says so. A copied test whose name was never changed does this. Give each test its own name:


In [42]:
%%writefile scratch/stations/tests/test_twice.py
from readings import to_fahrenheit


def test_freezing_point_is_32_fahrenheit():
    assert to_fahrenheit(0) == 32


def test_boiling_point_is_212_fahrenheit():
    assert to_fahrenheit(100) == 212


Overwriting scratch/stations/tests/test_twice.py


In [43]:
run_pytest("tests/test_twice.py", "--collect-only", "-q")


tests/test_twice.py::test_freezing_point_is_32_fahrenheit
tests/test_twice.py::test_boiling_point_is_212_fahrenheit

2 tests collected


### cannot collect test class 'TestReadings' because it has a __init__ constructor


In [44]:
%%writefile scratch/stations/tests/test_init.py
from readings import mean


class TestReadings:
    def __init__(self):
        self.readings = [4.2, 5.8]

    def test_mean_of_the_readings(self):
        assert mean(self.readings) == 5.0


Writing scratch/stations/tests/test_init.py


In [45]:
run_pytest("tests/test_init.py")


============================= test session starts ==============================
collected 0 items

=============================== warnings summary ===============================
tests/test_init.py:4
  tests/test_init.py:4: PytestCollectionWarning: cannot collect test class 'TestReadings' because it has a __init__ constructor (from: tests/test_init.py)
    class TestReadings:

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
============================== 1 warning ==============================


pytest makes an instance of a test class itself, with no arguments, and it will not guess what an
`__init__` needs, so it collected nothing from the class and warned. The run collected no tests at
all. The arrange step belongs in the test, and the **Fixtures** notebook shows how several tests
share one:


In [46]:
%%writefile scratch/stations/tests/test_init.py
from readings import mean


class TestReadings:
    def test_mean_of_the_readings(self):
        readings = [4.2, 5.8]

        assert mean(readings) == 5.0


Overwriting scratch/stations/tests/test_init.py


In [47]:
run_pytest("tests/test_init.py", "-q")


.                                                                        [100%]
1 passed


### TypeError: TestReadings.test_mean_of_two_readings() takes 0 positional arguments but 1 was given


In [48]:
%%writefile scratch/stations/tests/test_self.py
from readings import mean


class TestReadings:
    def test_mean_of_two_readings():
        assert mean([4.2, 5.8]) == 5.0


Writing scratch/stations/tests/test_self.py


In [49]:
report = pytest_report("tests/test_self.py", "-q")
print("\n".join(line for line in report.splitlines() if line.startswith(("E ", "1 failed"))))


E       TypeError: TestReadings.test_mean_of_two_readings() takes 0 positional arguments but 1 was given
1 failed


pytest calls a test method on an instance, which passes the instance as the first argument, and this
method has no parameter to receive it. The whole report runs through pytest's own code, with memory
addresses that change on every run, so the cell printed the `E` line and the counts. Give the method
`self`:


In [50]:
%%writefile scratch/stations/tests/test_self.py
from readings import mean


class TestReadings:
    def test_mean_of_two_readings(self):
        assert mean([4.2, 5.8]) == 5.0


Overwriting scratch/stations/tests/test_self.py


In [51]:
run_pytest("tests/test_self.py", "-q")


.                                                                        [100%]
1 passed


### import file mismatch


In [52]:
integration = PROJECT / "tests" / "integration"
integration.mkdir(exist_ok=True)
(integration / "test_summary.py").write_text(
    "from readings import summarize\n\n\ndef test_an_empty_day():\n    assert summarize([]) == {}\n")

run_pytest("-q")



==================================== ERRORS ====================================
____________________ ERROR collecting tests/test_summary.py ____________________
import file mismatch:
imported module 'test_summary' has this __file__ attribute:
  tests/integration/test_summary.py
which is not the same as the test file we want to collect:
  tests/test_summary.py
HINT: remove __pycache__ / .pyc files and/or use a unique basename for your test file modules
=========================== short test summary info ============================
ERROR tests/test_summary.py
!!!!!!!!!!!!!!!!!!!! Interrupted: 1 error during collection !!!!!!!!!!!!!!!!!!!!
1 error


`tests/integration/test_summary.py` and `tests/test_summary.py` have one name between them. pytest
imports a test file under its name alone, `test_summary`, when its folder has no `__init__.py`, and
Python can hold one module of a name at a time, so the second file with that name stops the run.
Give every test file a name of its own:


In [53]:
(integration / "test_summary.py").rename(integration / "test_summary_of_files.py")

run_pytest("-q")


.....................                                                    [100%]
21 passed


Last, the notebook is finished with its files, so this cell removes the scratch folder, with the
module, the `tests` folder and pytest's cache:


In [54]:
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- A test arranges a situation, acts with one call to the code under test, and asserts what the call
  produced. Cleanup undoes anything the test changed outside itself.
- Name a test for the behavior it checks, so that the first line of a failure says what broke.
- A test stops at its first failing `assert`, so keep it to one behavior, and one run reports every
  behavior that broke.
- Compare a whole value in one `assert` when you can: pytest reports which items differ and how many
  are the same.
- Keep tests in a `tests` folder, with a file for each part of the code and a name no other test file
  has, and group the tests of one function in a `Test` class with no `__init__`.
- Let every test arrange its own data, from a helper that returns a new value on every call, so that
  it passes alone and in any order.


## What is next

The **Fixtures** notebook moves the arrange step out of the test, into a function pytest runs for any
test that asks for it by name, and adds a cleanup step that runs after the test, whether it passed or
failed.


---

&#8592; **Previous:** [Your First Test](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/testing-and-packaging/02-your-first-test.ipynb)  &nbsp;·&nbsp;  [Testing and Packaging Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/testing-and-packaging.html)  &nbsp;·&nbsp;  **Next:** [Fixtures](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/testing-and-packaging/04-fixtures.ipynb) &#8594;
